In [1]:
# LangGraph에서 상태 그래프를 만들기 위한 클래스와
# 그래프의 시작/끝을 나타내는 상수를 가져옵니다.
from langgraph.graph import StateGraph, START, END

# 딕셔너리 형태의 타입(키/값 타입을 명시한 dict)을
# 정의할 수 있게 해주는 도구입니다.
from typing_extensions import TypedDict

# 타입에 "추가 정보(어노테이션)"를 붙일 수 있게 해주는 도구입니다.
# 여기서는 "이 필드는 어떤 방식으로 업데이트할지"를 적어줄 때 사용합니다.
from typing import Annotated

# 파이썬에 기본으로 들어있는 연산 함수들(더하기, 곱하기 등)을 모아둔 모듈입니다.
# 여기서는 리스트를 합칠 때 사용할 `operator.add` 를 씁니다.
import operator

In [2]:
# LangGraph 에서는 "리듀서(reducer) 함수"를 이용해서
# 같은 키의 값이 여러 번 나오면 어떻게 합칠지 정할 수 있습니다.
# 아래 예시는 단순히 old + new 로 합치는 함수입니다.
def update_function(old, new):
  return old + new

# 상태(State)에 어떤 키와 타입이 있는지 정의합니다.
class State(TypedDict):
  # messages 필드는 문자열 리스트(list[str]) 이고,
  # Annotated[..., operator.add] 를 사용해서
  #   "이 필드를 업데이트할 때는 operator.add 로 합쳐라" 라고 지정합니다.
  #   (리스트인 경우, 두 리스트를 이어 붙이는 효과가 납니다.)
  # messages: Annotated[list[str], update_function]
  messages: Annotated[list[str], operator.add]

# 위에서 정의한 State 타입을 사용하는 상태 그래프를 만듭니다.
graph_builder = StateGraph(State) 

# node_one 은 messages 리스트에 새 메시지를 하나 추가해서 반환합니다.
def node_one(state: State) -> State:
  return {
    "messages": ["Hello, nice to meet you. "]
  }

# node_two, node_three 는 여기서는 아무 것도 하지 않는 노드입니다.
# (빈 dict 를 반환하면, 상태가 그대로 유지됩니다.)
def node_two(state: State) -> State:
  return {}
  
def node_three(state: State) -> State:
  # print("node_three ->", state)
  return {}




In [3]:
# 그래프에 사용할 노드를 등록합니다.
# 첫 번째 인자: 그래프 안에서 부를 노드 이름(문자열)
# 두 번째 인자: 실제로 실행될 파이썬 함수
graph_builder.add_node("node_one", node_one)
graph_builder.add_node("node_two", node_two)
graph_builder.add_node("node_three", node_three)

# 노드들 사이의 실행 순서를 간선(edge)으로 정의합니다.
# START -> node_one -> node_two -> node_three -> END
# 이런 순서로 노드들이 차례대로 호출됩니다.
# 이때, 각 노드에서 반환한 state 들은
#   messages 필드에 대해서는 "리듀서 함수(operator.add)" 규칙에 따라
#   합쳐져서 최종 state 를 이룹니다.
graph_builder.add_edge(START, "node_one")
graph_builder.add_edge("node_one", "node_two")
graph_builder.add_edge("node_two", "node_three")
graph_builder.add_edge("node_three", END)




In [4]:
# 지금까지 정의한 노드와 상태, 간선 정보를 이용해서
# 실제로 실행 가능한 그래프 객체를 만듭니다.
graph = graph_builder.compile()

# graph.invoke(...) 를 호출하면 그래프를 "한 번 실행" 하는 것과 같습니다.
# - 초기 상태로 {"messages": ["Hello!!!"]} 를 넣어 줍니다.
# - START -> node_one -> node_two -> node_three -> END 순서로 노드가 실행되고,
# - messages 필드는 Annotated 에서 지정한 대로 operator.add 로 합쳐집니다.
#   => 최종 결과는 ["Hello!!!", "Hello, nice to meet you. "] 와 같이
#      초기 메시지와 node_one 에서 추가한 메시지가 이어 붙은 리스트가 됩니다.
graph.invoke(
  {
    "messages": ["Hello!!!"]
  }
)

{'messages': ['Hello!!!', 'Hello, nice to meet you. ']}